# CSCI461 – Assignment 1: Dockerized Data, Text Analysis and Machine Learning
## Dataset: SMS Spam Collection (UCI #228)

**Pipeline:** Dockerfile → custom image → container → JupyterLab → data cleaning → sampling → text analysis → TF‑IDF → classification + K‑Means → persistent outputs (`output/`, bind‑mounted to the host).

| Section | Content |
|---|---|
| 0 | Environment check (proves the notebook runs inside the container) |
| 4 | Data understanding, missing values & duplicates, `loc`/`iloc`, transformations, sampling, correlation |
| 5 | Text cleaning, tokenisation, word frequencies, spam vs ham comparison, insights |
| 6 | Supervised TF‑IDF classification (2 classifiers + tuned n‑gram model = bonus) and K‑Means clustering |
| 7 | Saved outputs checklist |

Run with **Kernel → Restart Kernel and Run All Cells**. Every random step uses `random_state=42`, so results are reproducible.

## 0. Setup and environment check

In [1]:
import os, re, sys, html, string, platform, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, Normalizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay, silhouette_score,
                             adjusted_rand_score, normalized_mutual_info_score)

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Project root: works whether Jupyter starts in bd-a1/ or in bd-a1/notebook/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_PATH = ROOT / "data" / "dataset.csv"
OUTPUT_DIR = ROOT / "output"
FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name):
    path = FIG_DIR / name
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"Saved figure -> {path.relative_to(ROOT)}")

in_docker = Path("/.dockerenv").exists() or os.environ.get("JUPYTER_DOCKER_STACKS_FOLDER") is not None \
            or os.environ.get("NB_USER") == "jovyan"
print("Running inside Docker container:", in_docker)
print("Hostname      :", platform.node())
print("Python        :", sys.version.split()[0])
print("pandas        :", pd.__version__, "| numpy:", np.__version__, "| scikit-learn:", sklearn.__version__)
print("Project root  :", ROOT)
print("Dataset path  :", DATA_PATH, "| exists:", DATA_PATH.exists())

Running inside Docker container: False
Hostname      : vm
Python        : 3.13.18
pandas        : 3.0.5 | numpy: 2.5.3 | scikit-learn: 1.9.1
Project root  : /home/claude/bd-a1
Dataset path  : /home/claude/bd-a1/data/dataset.csv | exists: True


## 4. Data Analysis and Preprocessing
### 4.1 Dataset understanding
The original UCI file (`SMSSpamCollection`) is tab‑separated with no header. It was converted once on the host into `data/dataset.csv` with two columns, `label` and `message` (see README for the exact command).

In [2]:
df_raw = pd.read_csv(DATA_PATH, encoding="utf-8")
df = df_raw.copy()
display(df.head(10))

,label,message
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives around here though"
5,spam,"FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, £1.50 to rcv"
6,ham,Even my brother is not like to speak with me. They treat me like aids patent.
7,ham,As per your request 'Melle Melle (Oru Minnaminunginte Nurungu Vettam)' has been set as your callertune for all Callers. Press *9 to copy your friends Callertune
8,spam,WINNER!! As a valued network customer you have been selected to receivea £900 prize reward! To claim call 09061701461. Claim code KL341. Valid 12 hours only.
9,spam,Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free! Call The Mobile Update Co FREE on 08002986030


In [3]:
print("Shape  :", df.shape)
print("Columns:", list(df.columns))
print("\nData types:\n", df.dtypes, sep="")
print()
df.info()

Shape  : (5574, 2)
Columns: ['label', 'message']

Data types:
label      str
message    str
dtype: object

<class 'pandas.DataFrame'>
RangeIndex: 5574 entries, 0 to 5573
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   label    5574 non-null   str  
 1   message  5574 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB


In [4]:
display(df.describe(include="all"))
print("\nClass distribution:")
display(pd.DataFrame({"count": df["label"].value_counts(),
                      "percent": df["label"].value_counts(normalize=True).mul(100).round(2)}))

,label,message
count,5574,5574
unique,2,5171
top,ham,"Sorry, I'll call later"
freq,4827,30



Class distribution:


,count,percent
label,,
ham,4827,86.6
spam,747,13.4


**What one row represents:** one SMS text message received on a mobile phone, together with a human‑assigned label saying whether it is legitimate (`ham`) or unsolicited advertising/scam (`spam`).

| Column type | Column(s) | Notes |
|---|---|---|
| **Text** | `message` | Raw SMS body (free text, slang, abbreviations, numbers, URLs) |
| **Label / category** | `label` | Binary target: `ham` (≈86.6 %) vs `spam` (≈13.4 %) → **imbalanced** |
| **Numerical** | *none in the raw file* | Numerical columns are **engineered** from the text in 4.4 (character count, word count, digit count, …) and used for correlation and normalisation |

`describe()` already shows that the most frequent message ("Sorry, I'll call later") occurs many times, a first hint of duplicates.

### 4.2 Missing values and duplicates

In [5]:
print("Missing values per column:")
display(df.isna().sum().to_frame("missing"))

empty_msgs = df["message"].fillna("").str.strip().eq("").sum()
print("Empty / whitespace-only messages:", empty_msgs)

dup_rows = df.duplicated().sum()
dup_msgs = df["message"].duplicated().sum()
conflicts = (df.groupby("message")["label"].nunique() > 1).sum()
print("Exact duplicate rows (label + message):", dup_rows)
print("Duplicate message texts              :", dup_msgs)
print("Same text with different labels      :", conflicts)

print("\nDuplicate rows by class:")
display(df[df.duplicated()]["label"].value_counts().to_frame("duplicates"))
print("Most repeated messages:")
display(df["message"].value_counts().head(5).to_frame("times"))

Missing values per column:


,missing
label,0
message,0


Empty / whitespace-only messages: 0
Exact duplicate rows (label + message): 403
Duplicate message texts              : 403
Same text with different labels      : 0

Duplicate rows by class:


,duplicates
label,
ham,309
spam,94


Most repeated messages:


,times
message,
"Sorry, I'll call later",30
I cant pick the phone right now. Pls send a message,12
Ok...,10
Please call our customer service representative on FREEPHONE 0808 145 4742 between 9am-11pm as you have WON a guaranteed £1000 cash or £5000 prize!,4
"Wen ur lovable bcums angry wid u, dnt take it seriously.. Coz being angry is d most childish n true way of showing deep affection, care n luv!.. kettoda manda... Have nice day da.",4


**Treatment and justification**

* **Missing values:** there are none (and no empty strings). A defensive `fillna("")` is still applied in the text‑cleaning function so the pipeline cannot break on future data.
* **Duplicates:** the exact duplicates are **removed** (keeping the first occurrence) because
  1. repeated spam templates and chain messages would be **over‑counted** in word frequencies and group statistics;
  2. if the same message appears in both the training and the test split, the classifier is tested on text it has **already seen** (data leakage), which inflates accuracy;
  3. no duplicate text carries conflicting labels, so dropping copies loses no information.

In [6]:
shape_before = df.shape
df = df.drop_duplicates(keep="first").reset_index(drop=True)
shape_after = df.shape
print(f"Shape before cleaning: {shape_before}")
print(f"Shape after cleaning : {shape_after}  ({shape_before[0] - shape_after[0]} rows removed)")
display(pd.DataFrame({"before": df_raw["label"].value_counts(), "after": df["label"].value_counts()}))

Shape before cleaning: (5574, 2)
Shape after cleaning : (5171, 2)  (403 rows removed)


,before,after
label,,
ham,4827,4518
spam,747,653
